In [0]:
customers = [
    (1, "Saravana", "Chennai", 30),
    (2, "John", "Bangalore", 28),
    (3, "Alice", "Hyderabad", 25),
    (3, "Alice", "Chennai", 25),
    (4, "David", None, 40),
    (5, None, "Chennai", 27),
    (6, "Kumar", "Chennai", -5),
    (7, "", "Bangalore", 32),
    (None, "Priya", "Chennai", 29)
]

columns = [
    "CustomerId",
    "CustomerName",
    "City",
    "Age"
]

df = spark.createDataFrame(customers, columns)

display(df)

CustomerId,CustomerName,City,Age
1,Saravana,Chennai,30
2,John,Bangalore,28
3,Alice,Hyderabad,25
3,Alice,Chennai,25
4,David,null,40
5,null,Chennai,27
6,Kumar,Chennai,-5
7,,Bangalore,32
null,Priya,Chennai,29


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze.day7_customers")

In [0]:
%sql
SELECT * FROM bronze.day7_customers;

CustomerId,CustomerName,City,Age
1,Saravana,Chennai,30
2,John,Bangalore,28
3,Alice,Hyderabad,25
3,Alice,Chennai,25
4,David,null,40
5,null,Chennai,27
6,Kumar,Chennai,-5
7,,Bangalore,32
null,Priya,Chennai,29


In [0]:
%sql
SELECT
    COUNT(*) AS TotalRecords,
    SUM(CASE WHEN CustomerId IS NULL THEN 1 ELSE 0 END) AS NullCustomerId,
    SUM(CASE WHEN CustomerName IS NULL THEN 1 ELSE 0 END) AS NullCustomerName,
    SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) AS NullCity,
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS NullAge
FROM bronze.day7_customers;

TotalRecords,NullCustomerId,NullCustomerName,NullCity,NullAge
9,1,1,1,0


In [0]:
%sql
SELECT
    CustomerId,
    COUNT(*) AS RecordCount
FROM bronze.day7_customers
GROUP BY CustomerId
HAVING COUNT(*) > 1;

CustomerId,RecordCount
3,2


In [0]:
%sql
SELECT *
FROM bronze.day7_customers
WHERE Age < 0
   OR Age > 120;

CustomerId,CustomerName,City,Age
6,Kumar,Chennai,-5


Rule 1:
CustomerId cannot be NULL

Rule 2:
CustomerId must be unique

Rule 3:
CustomerName cannot be NULL or empty

Rule 4:
City cannot be NULL

Rule 5:
Age must be between 0 and 120


In [0]:
from pyspark.sql.functions import col, trim
clean_df=(
    df.filter(col("CustomerId").isNotNull())
    .filter(col("CustomerName").isNotNull())
    .filter(col("City").isNotNull())
    .filter(trim(col("CustomerName")) != "")
    .filter(trim(col("City")) != "")
    .filter((col("Age") >= 0)& (col("Age") <= 120))
    .dropDuplicates(["CustomerId"])
    )
display(clean_df)

CustomerId,CustomerName,City,Age
1,Saravana,Chennai,30
2,John,Bangalore,28
3,Alice,Hyderabad,25


In [0]:
clean_df.write.format("delta").mode("overwrite").saveAsTable("silver.day7_customers")

In [0]:
%sql
SELECT *
FROM silver.day7_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age
1,Saravana,Chennai,30
2,John,Bangalore,28
3,Alice,Hyderabad,25


In [0]:
invalid_df = (
    df
    .filter(
        col("CustomerId").isNull()
        | col("CustomerName").isNull()
        | (trim(col("CustomerName")) == "")
        | col("City").isNull()
        | (col("Age") < 0)
        | (col("Age") > 120)
    )
)

display(invalid_df)

CustomerId,CustomerName,City,Age
4,David,null,40
5,null,Chennai,27
6,Kumar,Chennai,-5
7,,Bangalore,32
null,Priya,Chennai,29


In [0]:
invalid_df.write.format("delta").mode("overwrite").saveAsTable("silver.day7_customer_quarantine")

In [0]:
duplicate_Customer_Ids=(
    df.filter(
        col("CustomerId").isNotNull())
    .where(col("CustomerId").isNotNull())
    .groupBy("CustomerId")
    .count()
    .filter(col("count") > 1)
)
duplicate_df = (
    df
    .join(duplicate_Customer_Ids, on="CustomerId", how="inner")
    .drop("count")
)
display(duplicate_df)
duplicate_df.write.format("delta").mode("append").saveAsTable("silver.day7_customer_quarantine")

CustomerId,CustomerName,City,Age
3,Alice,Hyderabad,25
3,Alice,Chennai,25


In [0]:
%sql 
Select * from silver.day7_customer_quarantine;

CustomerId,CustomerName,City,Age
4,David,null,40
5,null,Chennai,27
6,Kumar,Chennai,-5
7,,Bangalore,32
null,Priya,Chennai,29
3,Alice,Hyderabad,25
3,Alice,Chennai,25
